# VecDB + OCI Embedding Demo
Walkthrough: call OCI Generative AI to create embeddings, persist them in Oracle VecDB, and keep the dataset refreshed for retrieval workflows.

## 1. Environment Setup
Install prerequisites and confirm you populated `.env` (or your environment) with the following variables:
- `VECDB_REST_URL`, `VECDB_USERNAME`, `VECDB_PASSWORD` – VecDB REST endpoint and connection credentials.
- `VECDB_TABLE` – target table for this demo, and `DROP_TABLE_WHEN_DONE` to let the cleanup cell decide whether to drop it.
- `OCI_PROFILE` and `OCI_REGION` – OCI CLI profile name and region used for the Generative AI request.
- `OCI_COMPARTMENT_OCID` – compartment that owns the Generative AI endpoint/model.
- `OCI_EMBED_MODEL`, `model_id`, and `OCI_GAI_ENDPOINT` – the hosted embedding model name, its OCID, and the inference endpoint URL.

Install the required Python dependencies in the following cell once per kernel.

This pipeline covers: (1) embedding text with OCI Generative AI, (2) persisting embeddings + metadata in Oracle VecDB, and (3) tracking updates via re-embedding. Use it as a template for retrieval-augmented generation workflows.

#### 🔧 Install Dependencies
Installs VecDB SDK (`oracle-vecdb`), OCI SDK (`oci`), `python-dotenv`, `pandas`, and `tqdm`. Run once per environment.

In [ ]:
%pip install -U oracle-vecdb oci python-dotenv pandas tqdm

## 2. Load Configuration
Read shared environment variables from `.env`. This keeps OCI and VecDB clients aligned across notebooks and CLIs. The assertion prevents accidental runs without VecDB credentials.

#### 🔑 Load Environment
Loads `.env` variables into memory and asserts VecDB credentials exist before continuing.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

OCI_PROFILE = os.getenv("OCI_PROFILE")
OCI_REGION = os.getenv("OCI_REGION")
resolved_host = os.getenv("VECDB_REST_URL")
VECDB_USER = os.getenv("VECDB_USERNAME")
VECDB_PASSWORD = os.getenv("VECDB_PASSWORD")
VECDB_ACCESS_TOKEN = os.getenv("VECDB_ACCESS_TOKEN")
OCI_COMPARTMENT_OCID = os.getenv("OCI_COMPARTMENT_OCID")
MODEL_ID = os.getenv("model_id")
VECDB_TABLE = os.getenv("VECDB_TABLE") or "OCI_EMBEDDINGS_DEMO"
EMBED_MODEL = os.getenv("OCI_EMBED_MODEL")
embed_endpoint = os.getenv("OCI_GAI_ENDPOINT")
assert resolved_host and (VECDB_ACCESS_TOKEN or (VECDB_USER and VECDB_PASSWORD)), "Missing VECDB connection values."
print(OCI_PROFILE, OCI_COMPARTMENT_OCID)


#### 🤝 Initialize Clients
Constructs the OCI Generative AI inference client and the Oracle VecDB client, honoring profile/region overrides and optional SSL relaxations.

In [ ]:
import oci
from oracle_vecdb import OracleVecDB, Configuration

config = oci.config.from_file(profile_name=OCI_PROFILE) if OCI_PROFILE else oci.config.from_file()
if OCI_REGION:
    config["region"] = OCI_REGION

signer = oci.signer.Signer(
    tenancy=config["tenancy"],
    user=config["user"],
    fingerprint=config["fingerprint"],
    private_key_file_location=config["key_file"],
    pass_phrase=oci.config.get_config_value_or_default(config, "pass_phrase"),
)

gai_client = oci.generative_ai_inference.GenerativeAiInferenceClient(config, signer=signer)


vec_config_kwargs = {"rest_url": resolved_host}
if VECDB_ACCESS_TOKEN:
    vec_config_kwargs["access_token"] = VECDB_ACCESS_TOKEN
else:
    vec_config_kwargs["username"] = VECDB_USER
    vec_config_kwargs["password"] = VECDB_PASSWORD
vec_config = Configuration(**vec_config_kwargs)
if os.getenv("VECDB_SELF_SIGNED_SSL", "false").lower() == "true":
    vec_config.verify_ssl = False
    import urllib3
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

vecdb = OracleVecDB(vec_config)
auth_method = 'bearer token' if vec_config.access_token else 'username/password'
print("OCI + VECDB clients ready")
print('Auth method:', auth_method)


## 3. Prepare Documents
Define a starter corpus, print the configured embedding model, and confirm we have predictable inputs.

#### 🗂️ Seed Sample Documents
The list contains `id` + `text` pairs. Swap with your domain payloads when adapting this notebook.

In [ ]:
documents = [
    {"id": "doc-001", "text": "OCI Generative AI provides multilingual embedding models."},
    {"id": "doc-002", "text": "Oracle VecDB accelerates semantic search workloads."},
    {"id": "doc-003", "text": "Combine OCI embeddings with VecDB for robust RAG pipelines."},
    {"id": "doc-004", "text": "VecDB indexing strategies impact query latency and freshness."},
    {"id": "doc-005", "text": "OCI policies govern access to Generative AI models."},
    {"id": "doc-006", "text": "Self-hosted inference endpoints support private networking."},
    {"id": "doc-007", "text": "VecDB metadata filters help segment tenant datasets."},
    {"id": "doc-008", "text": "Batch embedding jobs can be orchestrated with OCI data flows."},
    {"id": "doc-009", "text": "VecDB handles dense vectors alongside JSON metadata."},
    {"id": "doc-010", "text": "Refreshing embeddings keeps RAG answers grounded in recent data."},
]


print(EMBED_MODEL)
print(f"Prepared {len(documents)} documents")


### 4. Configure Inference Endpoint
Review the Generative AI endpoint (`OCI_GAI_ENDPOINT`) and model OCID we will use. Update `.env` to target private endpoints or different models.

#### 🌐 Configure Inference Endpoint
Echoes the endpoint/model being used and instantiates the Generative AI inference client with sample-aligned timeouts.

In [ ]:

retry_strategy = oci.retry.NoneRetryStrategy()
client_timeout = (10, 240)

gai_client = oci.generative_ai_inference.GenerativeAiInferenceClient(
    config=config,
    service_endpoint=embed_endpoint,
    retry_strategy=retry_strategy,
    timeout=client_timeout,
)
print('OCI Generative AI endpoint:', gai_client.base_client.endpoint)


### 5. Discover Models *(optional)*
List active Generative AI models in the compartment. 

In [ ]:
from oci.exceptions import ServiceError

try:
    models_client = oci.generative_ai.GenerativeAiClient(config=config, signer=signer)
    compartment_ocid=OCI_COMPARTMENT_OCID
    response = models_client.list_models(compartment_id=compartment_ocid, lifecycle_state='ACTIVE')
    print('Available models in compartment:')
    for item in response.data.items:
        model_type = getattr(item, 'model_type', getattr(item, 'category', 'unknown'))
        print(f"- {item.display_name} (type={model_type}): {item.id}")
except ServiceError as exc:
    print(f"Unable to list models ({exc.status} {exc.code}).")
    print(exc.message)
    print('Ensure the signer has policy to use generative-ai-family and the compartment OCID is correct.')


## 6. Persist Embeddings
Embed documents, provision a VecDB table, insert the vectors, and run a similarity query.

#### 🧮 Batch Embed Documents
Loops through each document, calling `embed_text` and handling both modern and legacy SDK response shapes while logging failures.

In [ ]:
from tqdm.auto import tqdm
from oci.exceptions import ServiceError

embed_results = []
for doc in tqdm(documents, desc='Embedding docs'):
    try:
        request = oci.generative_ai_inference.models.EmbedTextDetails(
            inputs=[doc['text']],
            serving_mode=oci.generative_ai_inference.models.OnDemandServingMode(model_id=MODEL_ID),
            compartment_id=OCI_COMPARTMENT_OCID,
            truncate='NONE',
        )
        response = gai_client.embed_text(request)
        payload = getattr(response, 'data', response)

        vector = None

        # payload.embeddings[0] as a list of floats
        if vector is None and getattr(payload, 'embeddings', None):
            first = payload.embeddings[0]
            vector = first if isinstance(first, list) else getattr(first, 'embedding', None)

        if not vector:
            print('Response attrs:', [a for a in dir(payload) if not a.startswith('_')])
            raise AttributeError('EmbedText response missing embedding payload.')

        embed_results.append({'id': doc['id'], 'dense_vector': vector, 'metadata': {'text': doc['text']}})

    except (ServiceError, AttributeError, IndexError) as exc:
        print(f"Embedding failed for {doc['id']} ({doc['text'][:40]}...): {getattr(exc, 'status', 'n/a')} {getattr(exc, 'code', type(exc).__name__)}")
        print(getattr(exc, 'message', str(exc)))
        raise

print('Generated embeddings for', len(embed_results), 'documents')


#### 🗄️ Persist Embeddings in VecDB
Creates a uniquely suffixed VecDB table for this run and bulk upserts the generated embeddings.

In [ ]:
target_table = VECDB_TABLE
try:
    vecdb.describe_vector_table(name=target_table)
    print(f"Table {target_table} already exists; using existing schema")
except Exception:
    print(f"Table {target_table} not found; creating it now")
    # Do not pass vector_index_params here. In this environment, minimal
    # vector index params such as {"auto_index": True/False} can trigger ORA-57715.
    vecdb.create_vector_table(
        name=target_table,
        table_params={"auto_generate_id": False},
        annotations={"text": "string"},
    )

if not embed_results:
    raise RuntimeError("No embeddings were generated; cannot upsert into VecDB.")

upsert_response = vecdb.upsert_vectors(table_name=target_table, vectors=embed_results)
print("Upsert response:", upsert_response)
TARGET_TABLE = target_table


#### 🔍 Similarity Query
Embeds an ad-hoc question via OCI and queries VecDB directly with the resulting vector, bypassing auto-embedding metadata.

In [ ]:
def query_items(response):
    if isinstance(response, list):
        return response
    if isinstance(response, tuple):
        return list(response)
    return getattr(response, "items", None) or getattr(response, "matches", None) or []


def result_metadata(item):
    return item.get("metadata", {}) if isinstance(item, dict) else getattr(item, "metadata", {})


def result_distance(item):
    if isinstance(item, dict):
        return item.get("distance")
    return getattr(item, "distance", getattr(item, "score", None))


def result_id(item):
    return item.get("id") if isinstance(item, dict) else getattr(item, "id", None)


def result_vector(item):
    if isinstance(item, dict):
        return item.get("vector") or item.get("dense_vector")
    return getattr(item, "vector", getattr(item, "dense_vector", None))


def result_text(item):
    return item.get("text", "") if isinstance(item, dict) else getattr(item, "text", "")


import pandas as pd

query_text = "How do I build RAG with Oracle?"

request = oci.generative_ai_inference.models.EmbedTextDetails(
    inputs=[query_text],
    serving_mode=oci.generative_ai_inference.models.OnDemandServingMode(model_id=MODEL_ID),
    compartment_id=OCI_COMPARTMENT_OCID,
    truncate='NONE',
)
response = gai_client.embed_text(request)
payload = getattr(response, 'data', response)

vector = None

if vector is None and getattr(payload, 'embeddings', None):
    first = payload.embeddings[0]
    vector = first if isinstance(first, list) else getattr(first, 'embedding', None)

if not vector:
    raise AttributeError("EmbedText response missing embedding payload.")

query = vecdb.query(
    table_name=TARGET_TABLE,
     query_by={"vector": vector},
    top_k=3,
    include_vectors=False,
)


pd.DataFrame([
    {
        'id': result_id(item),
        'score': result_distance(item),
        'text': result_metadata(item).get('text'),
    }
    
    for item in query_items(query)
])


## 8. Refresh Updated Documents
Show how to regenerate embeddings for a modified document and upsert it back into VecDB so downstream searches stay current.

#### ♻️ Refresh Updated Document
Shows how to re-embed a modified record and upsert it so VecDB stays synchronized with source changes.

In [ ]:
updated_doc = {"id": "doc-002", "text": "Oracle VecDB delivers high-performance vector search for enterprises."}

request = oci.generative_ai_inference.models.EmbedTextDetails(
    inputs=[updated_doc["text"]],
    serving_mode=oci.generative_ai_inference.models.OnDemandServingMode(model_id=MODEL_ID),
    compartment_id=OCI_COMPARTMENT_OCID,
)
new_vector = gai_client.embed_text(request)
payload = getattr(new_vector, 'data', new_vector)

vector = None

if getattr(payload, 'embeddings', None):
    first = payload.embeddings[0]
    vector = first if isinstance(first, list) else getattr(first, 'embedding', None)

if not vector:
    raise AttributeError("EmbedText response missing embedding payload.")

refresh_payload = [{"id": updated_doc["id"], "dense_vector": vector, "metadata": {"text": updated_doc["text"]}}]
vecdb.upsert_vectors(table_name=TARGET_TABLE, vectors=refresh_payload)
print("Refreshed embeddings for", updated_doc["id"])


## 9. Cleanup (Optional)
Drop the demo vector table when you’re finished exploring. Leave it in place if you want to reuse the embeddings across sessions.

In [ ]:
if os.getenv("DROP_TABLE_WHEN_DONE", "false").lower() == "true":
    vecdb.drop_vector_table(name=TARGET_TABLE)
    print(f'Dropped table {TARGET_TABLE}')
else:
    print('Cleanup skipped. Set DROP_TABLE_WHEN_DONE=true to enable table drop.')
